# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")
openai = OpenAI(base_url = "https://generativelanguage.googleapis.com/v1beta/openai/", api_key = google_api_key)
MODEL = "gemini-2.5-flash-lite"

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [01:09<00:00, 23.05s/it]


In [5]:
len(deals)

30

In [6]:
deals[10].describe()

'Title: Lenovo IdeaPad Slim 3 Ryzen 5 7520U 15.6" Touch Laptop for $294 + free shipping\nDetails: Use promo code "VIPOUTLETAPR26" to drop the price. That\'s more than $200 off list. It\'s also the best price you\'d find on this model anywhere today. Buy Now at eBay\nFeatures: AMD Ryzen 5 7520U 2.8GHz 15.6" 1920x1080 (1080p) Touch display 8GB RAM & 512GB SSD AMD Radeon 610M Graphics\nURL: https://www.dealnews.com/products/Lenovo/Lenovo-Idea-Pad-Slim-3-Ryzen-5-7520-U-15-6-Touch-Laptop/498256.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [7]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [8]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [9]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Apple Watch Series 9 45mm GPS Smartwatch for $154 + free shipping
Details: Use promo code "VIPOUTLETAPR26" to drop the price. That's by far the best we've seen in any condition. Shipping is free. Buy Now at eBay
Features: always-on Retina display  up to 18 hours battery life  blood oxygen & ECG apps Model: MR9A3LL/A
URL: https://www.dealnews.com/products/Apple/Apple-Watch-Serie

In [10]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='This Samsung 65-inch 4K UHD Smart TV features a 3840x2160 resolution and a 60Hz refresh rate, supporting HDR10+ for enhanced visual quality. It runs on the Tizen OS and includes built-in Amazon Alexa and Bixby voice assistants. Connectivity options include 2 HDMI ports, 1 USB-A port, and 1 Ethernet port.', price=245.0, url='https://www.dealnews.com/products/Samsung/Samsung-U7900-Series-UN65-U7900-FFXZA-65-4-K-UHD-Smart-TV/497591.html?iref=rss-c142'), Deal(product_description='The Hisense 85-inch U8 Series Mini-LED ULED Smart TV delivers a stunning 85-inch Mini-LED display with a native 165Hz refresh rate and AMD FreeSync Premium Pro. It supports up to 5,000 nits peak brightness and features an integrated 4.1.2 channel audio system for an immersive viewing experience.', price=1500.0, url='https://www.dealnews.com/products/Hisense/Hisense-U8-Series-85-U8-QG-85-Mini-LED-ULED-4-K-Smart-TV/498255.html?iref=rss-c142'), Deal(product_description='

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


This Samsung 65-inch 4K UHD Smart TV features a 3840x2160 resolution and a 60Hz refresh rate, supporting HDR10+ for enhanced visual quality. It runs on the Tizen OS and includes built-in Amazon Alexa and Bixby voice assistants. Connectivity options include 2 HDMI ports, 1 USB-A port, and 1 Ethernet port.
245.0
https://www.dealnews.com/products/Samsung/Samsung-U7900-Series-UN65-U7900-FFXZA-65-4-K-UHD-Smart-TV/497591.html?iref=rss-c142

The Hisense 85-inch U8 Series Mini-LED ULED Smart TV delivers a stunning 85-inch Mini-LED display with a native 165Hz refresh rate and AMD FreeSync Premium Pro. It supports up to 5,000 nits peak brightness and features an integrated 4.1.2 channel audio system for an immersive viewing experience.
1500.0
https://www.dealnews.com/products/Hisense/Hisense-U8-Series-85-U8-QG-85-Mini-LED-ULED-4-K-Smart-TV/498255.html?iref=rss-c142

This Vizio 43-inch 4K HDR LED Smart TV, model V4K43M-08, offers a 3840x2160 (4K) resolution for sharp and clear picture quality. It

In [3]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [4]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [4]:
result

DealSelection(deals=[Deal(product_description='This 65-inch Samsung TV boasts a 4K UHD resolution with a 60Hz refresh rate and HDR10+ support, delivering vibrant and sharp visuals. It runs on the Tizen OS, featuring built-in Amazon Alexa and Bixby for smart home integration. Connectivity options include two HDMI ports, one USB-A port, and an Ethernet port.', price=245.0, url='https://www.dealnews.com/products/Samsung/Samsung-U7900-Series-UN65-U7900-FFXZA-65-4-K-UHD-Smart-TV/497591.html?iref=rss-c142'), Deal(product_description='Experience premium home entertainment with this 85-inch Hisense ULED 4K Smart TV from the U8 Series. It features a Mini-LED display for exceptional brightness and contrast, a native 165Hz refresh rate for smooth motion, and AMD FreeSync Premium Pro. The TV is capable of up to 5,000 nits peak brightness and includes a 4.1.2 channel audio system.', price=1500.0, url='https://www.dealnews.com/products/Hisense/Hisense-U8-Series-85-U8-QG-85-Mini-LED-ULED-4-K-Smart-TV

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [13]:
load_dotenv(override=True)

True

In [14]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [15]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [16]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [17]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [18]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
23:43:13 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= gemma4:e4b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= gemma4:e4b; provider = ollama
23:43:53 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
